# Kashmir Birds AI — baseline training (single flat model)

**Model:** ConvNeXt-Tiny, pretrained on ImageNet, fine-tuned on 691 Kashmir bird species (one flat 691-way classifier). This is the baseline to beat — the two-stage family-first variant is tested against it afterwards, on the same test set.

**Where to run:** Kaggle notebook, free T4 GPU (16 GB VRAM). Turn GPU on: Settings → Accelerator → GPU T4.
**Do NOT run this until the dataset is uploaded** — see the next cell.

**Dataset:** `dataset/` with `train/val/test/` folders, one folder per species named `<Scientific_name>` (spaces → `_`). Split is already done **by observation** (70/15/15) — do not re-split.
**Expected size:** ~70,000–100,000 images, ~10–15 GB.


## 1. Upload the dataset to Kaggle (one-time, do this first)

1. On your computer, zip the folder `~/workspace/kashmir-birds/model-data/dataset/` **and** `cleaned-species-list.csv` together.
   - The zip must contain a top-level `dataset/` folder with `train/val/test/` inside it, plus `cleaned-species-list.csv` at the top level.
2. On Kaggle: **Create → New Dataset** → upload the zip → name it `kashmir-birds-dataset` → Create.
3. In **this** notebook: **Add data → Your Datasets → kashmir-birds-dataset → Add**.
4. Also upload `train_baseline.py` (in the same folder as this notebook) as a notebook file, or paste it into the cell in step 3.
5. Check the path in the next cell prints the expected counts before training.


In [ ]:
import os, pathlib
data = pathlib.Path('/kaggle/input/kashmir-birds-dataset/dataset')
print('dataset exists:', data.exists())
for split in ('train','val','test'):
    p = data/split
    n_cls = len(list(p.iterdir())) if p.exists() else 0
    n_img = sum(1 for _ in p.rglob('*.jpg')) + sum(1 for _ in p.rglob('*.png')) if p.exists() else 0
    print(split, 'classes:', n_cls, 'images:', n_img)
print('cleaned list:', os.path.exists('/kaggle/input/kashmir-birds-dataset/cleaned-species-list.csv'))


## 2. Install dependencies

Torch is preinstalled on Kaggle. `timm` usually is too — this cell makes sure, and is safe to re-run.


In [ ]:
!pip install -q timm==1.0.15 2>&1 | tail -2
import torch, timm
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('timm', timm.__version__)


## 3. Train the baseline

This runs `train_baseline.py` (in the notebook's working directory). It does, in order:
- **Phase 1 (3 epochs):** classifier head only, backbone frozen, LR 3e-4
- **Phase 2 (17 epochs):** whole model unfrozen, LR 5e-5, cosine schedule
- Mixed precision, batch 64, RandAugment, MixUp/CutMix, label smoothing 0.1, class-balanced sampling
- Saves the best checkpoint (by val top-1) to `./outputs/best_checkpoint.pth`

**Time estimate on a T4:** roughly 3–6 hours for 20 epochs on ~70k images. If the session is interrupted, re-run from the last saved checkpoint — the script always saves the best-so-far.
If you are on a 6 GB laptop GPU instead, use `--batch 32 --model convnext_tiny`.


In [ ]:
!python train_baseline.py --data /kaggle/input/kashmir-birds-dataset/dataset --species-csv /kaggle/input/kashmir-birds-dataset/cleaned-species-list.csv --out ./outputs --batch 64 --epochs 20


## 4. Results

`train_baseline.py` already prints and saves `outputs/metrics.json` (val best top-1/top-5, test top-1/top-5 with flip TTA, worst genera). This cell just displays it, plus the worst genera — those are the lookalike groups to attack next.


In [ ]:
import json, pathlib
m = json.loads(pathlib.Path('./outputs/metrics.json').read_text())
print(json.dumps(m, indent=2))


## 5. Export for the website

The script already exported `outputs/model.onnx` (plus `classes.json`). Download both from the notebook's output files and put them in the Kashmir Birds website — the AI Identify page (currently "Coming Soon") will load `model.onnx` with ONNX Runtime and `classes.json` for the species names. **Do not commit the ONNX file to the repo until it has been tested locally** — it is ~100 MB.


In [ ]:
import os
for f in ('best_checkpoint.pth','model.onnx','classes.json','metrics.json'):
    p = f'./outputs/{f}'
    print(f, os.path.exists(p), f'{os.path.getsize(p)/1e6:.1f} MB' if os.path.exists(p) else '')


## Next step (only after this baseline is measured)

Test the two-stage variant on the **same test set**: Model 1 = family classifier (89 families), Model 2 = the species model above, gated to the top-3 predicted families. Keep whichever wins on test top-1/top-5 — do not keep two models just because the idea sounds better.
